# Will this order be late? Dashboard

`uv run python main.py app` runs this notebook. It trains nothing. It loads what `train.ipynb` saved in `artifacts/`:

- the fitted preprocessor and the three models, which score every order again here
- `predictions.csv`: every order with its split, features and true `late` flag
- `metadata.json`: the thresholds, the costs and the split dates

1. Load
2. Helpers
3. Dashboard
4. Launch

## 1. Load

In [ ]:
import json
from pathlib import Path

import gradio as gr
import joblib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch
import torch.nn as nn

ARTIFACTS = Path("artifacts")
if not (ARTIFACTS / "predictions.csv").exists():
    raise SystemExit("artifacts/ not found. Run `uv run python main.py train` first.")

preds = pd.read_csv(ARTIFACTS / "predictions.csv", parse_dates=["purchase_ts"])
meta = json.loads((ARTIFACTS / "metadata.json").read_text())


# the saved preprocessor and models (loaded, never refitted)
preprocessor = joblib.load(ARTIFACTS / "preprocessor.joblib")
sk_model = joblib.load(ARTIFACTS / "sklearn_model.joblib")
manual = torch.load(ARTIFACTS / "manual_pytorch.pt")


class LogisticModel(nn.Module):   # same class as in train.ipynb, needed to load the state_dict
    def __init__(self, n_features):
        super().__init__()
        self.linear = nn.Linear(n_features, 1)

    def forward(self, X):
        return self.linear(X).squeeze(1)


module_model = LogisticModel(meta["n_features"])
module_model.load_state_dict(torch.load(ARTIFACTS / "module_pytorch.pt"))
module_model.eval()

# score every order with the loaded models
X = preprocessor.transform(preds[meta["feature_cols"]])
X_tensor = torch.tensor(X, dtype=torch.float32)
with torch.no_grad():
    preds["prob_sklearn"] = sk_model.predict_proba(X)[:, 1]
    preds["prob_manual"] = torch.sigmoid(X_tensor @ manual["w"] + manual["b"]).numpy()
    preds["prob_module"] = torch.sigmoid(module_model(X_tensor)).numpy()

# display name -> probability column in predictions.csv
MODELS = {"sklearn": "prob_sklearn", "manual pytorch": "prob_manual", "nn.Module": "prob_module"}
COLORS = {"sklearn": "#2a78d6", "manual pytorch": "#eb6834", "nn.Module": "#1baf7a"}
LATE_COLOR, ON_TIME_COLOR = "#eb6834", "#2a78d6"
SPLITS = ["test", "val", "train"]
NUMERIC = meta["numeric_cols"]
CATEGORICAL = meta["categorical_cols"]

FEATURE_LABELS = {
    "n_sellers": "Number of sellers",
    "promised_days": "Promised delivery time (days)",
    "log_distance_km": "log(1 + seller-to-customer distance in km)",
    "log_freight": "log(1 + freight cost)",
    "customer_state": "Customer state",
}

print(preds.groupby("split").size().to_dict())

## 2. Helpers

### Cost of a threshold

Same cost as in `train.ipynb`: a missed late order costs `cost_fn`, a false alarm costs `cost_fp`. The dashboard starts with the values from `metadata.json`, and both can be changed in the threshold tab.

In [ ]:
def split_df(split):
    return preds[preds["split"] == split]


def confusion(y_true, y_prob, threshold):
    y_pred = np.asarray(y_prob) >= threshold
    y_true = np.asarray(y_true).astype(bool)
    tp = int((y_pred & y_true).sum())
    fp = int((y_pred & ~y_true).sum())
    fn = int((~y_pred & y_true).sum())
    tn = int((~y_pred & ~y_true).sum())
    return tp, fp, fn, tn


def total_cost(y_true, y_prob, threshold, cost_fn, cost_fp):
    tp, fp, fn, tn = confusion(y_true, y_prob, threshold)
    return cost_fn * fn + cost_fp * fp


def style(fig, title, height=430):
    fig.update_layout(
        title=dict(text=title, x=0.01, font=dict(size=15)),
        template="plotly_white",
        height=height,
        margin=dict(l=60, r=20, t=60, b=55),
        legend=dict(orientation="h", y=-0.2, x=0),
        hoverlabel=dict(font_size=12),
    )
    return fig

### Prediction vs actual

The target is 0 or 1, so a plain scatter of prediction against actual would just be two lines of dots. Instead I sort the orders by predicted probability, cut them into 10 equal groups and plot the average prediction (x) against the share that was really late (y). On the diagonal, the model's probabilities are right. If the three models' lines sit on top of each other, they agree.

In [ ]:
def calibration_fig(split, n_bins=10):
    d = split_df(split)
    fig = go.Figure()
    top = 0.0
    for name, col in MODELS.items():
        bins = pd.qcut(d[col].rank(method="first"), n_bins, labels=False)
        g = d.groupby(bins).agg(pred=(col, "mean"), actual=("late", "mean"), n=("late", "size"))
        top = max(top, g.pred.max(), g.actual.max())
        fig.add_trace(go.Scatter(
            x=g.pred, y=g.actual, mode="lines+markers", name=name,
            line=dict(color=COLORS[name], width=2), marker=dict(size=9, line=dict(width=2, color="white")),
            customdata=g.n,
            hovertemplate=f"{name}<br>mean predicted %{{x:.3f}}<br>actually late %{{y:.3f}}<br>%{{customdata}} orders<extra></extra>",
        ))
    top *= 1.1
    fig.add_trace(go.Scatter(x=[0, top], y=[0, top], mode="lines", name="perfect calibration",
                             line=dict(color="#8a8a85", dash="dash", width=1)))
    fig.update_xaxes(title="Mean predicted probability of late", range=[0, top])
    fig.update_yaxes(title="Share actually late", range=[0, top])
    return style(fig, f"Predicted vs actual late rate, {split} split ({len(d):,} orders, {n_bins} equal-sized bins)")


def agreement_fig(split):
    d = split_df(split)
    fig = go.Figure()
    for name in ["manual pytorch", "nn.Module"]:
        fig.add_trace(go.Scattergl(
            x=d["prob_sklearn"], y=d[MODELS[name]], mode="markers", name=f"{name} vs sklearn",
            marker=dict(size=4, color=COLORS[name], opacity=0.5),
            hovertemplate="sklearn %{x:.4f}<br>" + name + " %{y:.4f}<extra></extra>",
        ))
    top = float(d[list(MODELS.values())].to_numpy().max()) * 1.05
    fig.add_trace(go.Scatter(x=[0, top], y=[0, top], mode="lines", name="identical",
                             line=dict(color="#8a8a85", dash="dash", width=1)))
    fig.update_xaxes(title="sklearn probability")
    fig.update_yaxes(title="PyTorch probability")
    max_diff = max(np.abs(d["prob_sklearn"] - d[c]).max() for c in ("prob_manual", "prob_module"))
    return style(fig, f"Do the three implementations agree? Largest difference: {max_diff:.4f}")


def results_table(split):
    d = split_df(split)
    rows = []
    for name, col in MODELS.items():
        t = meta["threshold_by_model"][name]
        tp, fp, fn, tn = confusion(d["late"], d[col], t)
        rows.append({
            "model": name, "threshold": t,
            "precision": round(tp / (tp + fp), 3) if tp + fp else 0.0,
            "recall": round(tp / (tp + fn), 3) if tp + fn else 0.0,
            "tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "cost": meta["cost_fn"] * fn + meta["cost_fp"] * fp,
        })
    n_late = int(d["late"].sum())
    n_ok = len(d) - n_late
    rows.append({"model": "never late", "threshold": None, "precision": 0.0, "recall": 0.0,
                 "tp": 0, "fp": 0, "fn": n_late, "tn": n_ok, "cost": meta["cost_fn"] * n_late})
    rows.append({"model": "always late", "threshold": None,
                 "precision": round(n_late / len(d), 3), "recall": 1.0,
                 "tp": n_late, "fp": n_ok, "fn": 0, "tn": 0, "cost": meta["cost_fp"] * n_ok})
    return pd.DataFrame(rows)

### Threshold and cost

The confusion matrix and the cost are recomputed from the probabilities every time the slider moves. The cost curve shows the cost at every threshold for one model, with the two naive baselines as flat lines. The model is only worth using where its curve is below both.

In [ ]:
def confusion_fig(tp, fp, fn, tn):
    z = [[tn, fp], [fn, tp]]
    fig = go.Figure(go.Heatmap(
        z=z, x=["Predicted on time", "Predicted late"], y=["Actually on time", "Actually late"],
        text=[[f"{v:,}" for v in row] for row in z], texttemplate="%{text}", textfont=dict(size=20),
        colorscale=[[0, "#f1f6fc"], [1, "#2a78d6"]], showscale=False, xgap=2, ygap=2,
        hovertemplate="%{y}, %{x}: %{z:,}<extra></extra>",
    ))
    fig.update_yaxes(autorange="reversed")
    return style(fig, "Confusion matrix", height=360)


def cost_curve_fig(split, model, threshold, cost_fn, cost_fp):
    d = split_df(split)
    ts = np.arange(0.01, 0.61, 0.01)
    costs = [total_cost(d["late"], d[MODELS[model]], t, cost_fn, cost_fp) for t in ts]
    n_late = int(d["late"].sum())
    never = cost_fn * n_late
    always = cost_fp * (len(d) - n_late)
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=ts, y=costs, mode="lines", name=model,
                             line=dict(color=COLORS[model], width=2),
                             hovertemplate="threshold %{x:.2f}<br>cost %{y:,.0f}<extra></extra>"))
    fig.add_hline(y=never, line=dict(color="#8a8a85", dash="dash", width=1),
                  annotation_text=f"never late: {never:,.0f}", annotation_position="top right")
    fig.add_hline(y=always, line=dict(color="#8a8a85", dash="dot", width=1),
                  annotation_text=f"always late: {always:,.0f}", annotation_position="bottom right")
    current = total_cost(d["late"], d[MODELS[model]], threshold, cost_fn, cost_fp)
    fig.add_trace(go.Scatter(x=[threshold], y=[current], mode="markers", name="current threshold",
                             marker=dict(size=12, color=COLORS[model], line=dict(width=2, color="white")),
                             hovertemplate="threshold %{x:.2f}<br>cost %{y:,.0f}<extra></extra>"))
    fig.update_xaxes(title="Decision threshold")
    fig.update_yaxes(title="Total business cost")
    return style(fig, f"Cost across thresholds, {model}, {split} split")


def threshold_view(split, model, threshold, cost_fn, cost_fp):
    d = split_df(split)
    y, p = d["late"], d[MODELS[model]]
    tp, fp, fn, tn = confusion(y, p, threshold)
    cost = cost_fn * fn + cost_fp * fp
    n_late = tp + fn
    never = cost_fn * n_late
    always = cost_fp * (len(d) - n_late)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    verdict = "cheaper" if cost < min(never, always) else "NOT cheaper"
    summary = (
        f"### Total cost: {cost:,.0f}\n"
        f"{fn:,} missed late orders x {cost_fn:g} + {fp:,} false alarms x {cost_fp:g}\n\n"
        f"At this threshold the model is **{verdict}** than the best naive baseline "
        f"(never late: {never:,.0f}, always late: {always:,.0f}).\n\n"
        f"Precision {precision:.1%}, recall {recall:.1%}, flagged {tp + fp:,} of {len(d):,} orders."
    )
    return confusion_fig(tp, fp, fn, tn), cost_curve_fig(split, model, threshold, cost_fn, cost_fp), summary

### Distributions

Numeric features: histograms of late and on-time orders on top of each other. Customer state: late rate per state. The target tab also shows the late rate per month, which is where the change between train, validation and test shows up.

In [ ]:
def feature_fig(feature, split):
    d = split_df(split)
    label = FEATURE_LABELS.get(feature, feature)
    fig = go.Figure()
    if feature in CATEGORICAL:
        g = d.groupby(feature).agg(rate=("late", "mean"), n=("late", "size")).sort_values("rate", ascending=False)
        fig.add_trace(go.Bar(x=g.index, y=g.rate, customdata=g.n, marker_color=ON_TIME_COLOR, name="late rate",
                             hovertemplate="%{x}<br>late rate %{y:.1%}<br>%{customdata:,} orders<extra></extra>"))
        fig.add_hline(y=d["late"].mean(), line=dict(color="#8a8a85", dash="dash", width=1),
                      annotation_text=f"overall {d['late'].mean():.1%}", annotation_position="top right")
        fig.update_yaxes(title="Late rate", tickformat=".0%")
        fig.update_xaxes(title=label)
        return style(fig, f"Late rate by {label.lower()}, {split} split")
    for flag, name, color in [(0, "on time", ON_TIME_COLOR), (1, "late", LATE_COLOR)]:
        fig.add_trace(go.Histogram(x=d.loc[d["late"] == flag, feature], name=name, marker_color=color,
                                   opacity=0.6, histnorm="probability",
                                   hovertemplate="%{x}<br>share of " + name + " orders %{y:.1%}<extra></extra>"))
    fig.update_layout(barmode="overlay")
    fig.update_xaxes(title=label)
    fig.update_yaxes(title="Share of orders in group", tickformat=".0%")
    return style(fig, f"{label}: late vs on-time orders, {split} split")


def target_fig():
    d = preds.assign(month=preds["purchase_ts"].dt.to_period("M").dt.to_timestamp())
    g = d.groupby("month").agg(rate=("late", "mean"), n=("late", "size")).reset_index()
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=g.month, y=g.rate, mode="lines+markers", name="late rate",
                             line=dict(color=LATE_COLOR, width=2), customdata=g.n,
                             marker=dict(size=8, line=dict(width=2, color="white")),
                             hovertemplate="%{x|%b %Y}<br>late rate %{y:.1%}<br>%{customdata:,} orders<extra></extra>"))
    for label, start in [("validation starts", meta["val_start"]), ("test starts", meta["test_start"])]:
        fig.add_vline(x=pd.Timestamp(start).timestamp() * 1000, line=dict(color="#8a8a85", dash="dash", width=1),
                      annotation_text=label, annotation_position="top left")
    fig.update_xaxes(title="Purchase month")
    fig.update_yaxes(title="Late rate", tickformat=".0%", rangemode="tozero")
    return style(fig, "Late rate by purchase month (train, then validation, then test)")


def split_summary():
    g = preds.groupby("split").agg(orders=("late", "size"), late_rate=("late", "mean"),
                                   first=("purchase_ts", "min"), last=("purchase_ts", "max"))
    g = g.loc[["train", "val", "test"]].reset_index()
    g["late_rate"] = (g["late_rate"] * 100).round(1).astype(str) + "%"
    g["first"], g["last"] = g["first"].dt.date, g["last"].dt.date
    return g

## 3. Dashboard

Three tabs:

1. Compare models: prediction vs actual, how close the three models are, and the results table with the baselines.
2. Threshold and cost: slider, confusion matrix and business cost.
3. Distributions: features and target.

The split picker starts on test (the newest months). Train is there too, but its scores look better than they are because the models were fitted on it.

In [ ]:
default_model = "sklearn"
default_threshold = meta["threshold_by_model"][default_model]

with gr.Blocks(title="Late delivery risk dashboard") as demo:
    gr.Markdown(
        "# Will this order be late?\n"
        "Logistic regression trained three ways (scikit-learn, manual PyTorch, `nn.Module`) on Olist orders. "
        "Everything here is read from the saved outputs of the training stage; nothing is retrained."
    )
    split = gr.Radio(SPLITS, value="test", label="Evaluation split (test = held-out future orders)")

    with gr.Tab("Compare models"):
        calib = gr.Plot(label="Prediction vs actual")
        agree = gr.Plot(label="Agreement")
        gr.Markdown(
            "**Results at each model's tuned threshold.** Cost uses "
            f"{meta['cost_fn']:g} per missed late order and {meta['cost_fp']:g} per false alarm. "
            "`never late` and `always late` are the naive baselines."
        )
        table = gr.Dataframe(interactive=False)

    with gr.Tab("Threshold and cost"):
        with gr.Row():
            model = gr.Dropdown(list(MODELS), value=default_model, label="Model")
            threshold = gr.Slider(0.01, 0.99, value=default_threshold, step=0.01,
                                  label="Decision threshold (flag order as late if probability >= threshold)")
        with gr.Row():
            cost_fn = gr.Number(value=meta["cost_fn"], label="Cost of a missed late order", minimum=0)
            cost_fp = gr.Number(value=meta["cost_fp"], label="Cost of a false alarm", minimum=0)
        summary = gr.Markdown()
        with gr.Row():
            cm_plot = gr.Plot(label="Confusion matrix")
            cost_plot = gr.Plot(label="Cost curve")

    with gr.Tab("Distributions"):
        feature = gr.Dropdown(NUMERIC + CATEGORICAL, value=NUMERIC[0], label="Feature",
                              info="Uses the evaluation split chosen above")
        feat_plot = gr.Plot(label="Feature distribution")
        target_plot = gr.Plot(label="Target over time")
        gr.Markdown("**The three splits** (time-based: train is oldest, test is newest)")
        gr.Dataframe(value=split_summary(), interactive=False)

    # --- wiring ---
    split.change(calibration_fig, split, calib)
    split.change(agreement_fig, split, agree)
    split.change(results_table, split, table)

    thr_inputs = [split, model, threshold, cost_fn, cost_fp]
    thr_outputs = [cm_plot, cost_plot, summary]
    for control in (split, model, threshold, cost_fn, cost_fp):
        control.change(threshold_view, thr_inputs, thr_outputs)
    # picking another model jumps the slider to that model's tuned threshold
    model.change(lambda m: meta["threshold_by_model"][m], model, threshold)

    split.change(feature_fig, [feature, split], feat_plot)
    feature.change(feature_fig, [feature, split], feat_plot)

    def render_all(s, m, t, cfn, cfp, f):
        return (calibration_fig(s), agreement_fig(s), results_table(s),
                *threshold_view(s, m, t, cfn, cfp), feature_fig(f, s), target_fig())

    demo.load(render_all, [split, model, threshold, cost_fn, cost_fp, feature],
              [calib, agree, table, cm_plot, cost_plot, summary, feat_plot, target_plot])

## 4. Launch

`main.py` turns this notebook into a script and runs it, so `demo.launch()` keeps the app running on port 7860.

In [ ]:
demo.launch()